In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [6]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!wget $data -O course_lead_scoring_2026.csv

/bin/bash: wget: command not found


In [28]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.shape

(5000, 9)

In [8]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [9]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [10]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [11]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

In [12]:
df_cat = df[['lead_source', 'industry', 'employment_status', 'location']]
df_num = df[['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted']]

In [13]:
df_cat.fillna('NA', inplace=True)
df_num.fillna(0.0, inplace=True)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,88160.0,3,4,0.64,1
1,72688.0,3,7,0.72,1
2,44697.0,3,4,0.58,1
3,0.0,3,6,0.57,0
4,24062.0,4,5,0.62,1
...,...,...,...,...,...
4995,45834.0,2,2,0.37,1
4996,84831.0,2,5,0.51,0
4997,53636.0,4,6,0.75,1
4998,69430.0,4,8,0.79,1


Question 1

In [14]:
df_cat['industry'].mode()

0    technology
Name: industry, dtype: str

The most frequent observation is "Techonology"

Question 2

In [15]:
df_num.corrwith(df_num['converted']).sort_values(ascending=False)

converted                   1.000000
lead_score                  0.483246
interaction_count           0.448624
number_of_courses_viewed    0.395038
annual_income               0.189652
dtype: float64

In [16]:
df_num_corr = df_num.corr()
display(df_num_corr)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


The highest correlation is between 'lead_score' and 'interaction_count'

In [19]:
df_train_prep = pd.concat([df_num, df_cat], axis=1)

In [29]:

df_full_train, df_test = train_test_split(df_train_prep, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [32]:
df_train.head()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,lead_source,industry,employment_status,location
0,65980.0,2,3,0.37,paid_ads,finance,employed,asia
1,71662.0,0,0,0.09,events,finance,employed,south_america
2,40220.0,2,5,0.55,referral,NA,employed,asia
3,88352.0,2,2,0.45,organic_search,finance,employed,north_america
4,89316.0,1,5,0.00,events,technology,employed,north_america


In [31]:
df_train.reset_index(drop=True, inplace=True)
display(df_train.head(10))

,annual_income,number_of_courses_viewed,interaction_count,lead_score,lead_source,industry,employment_status,location
0,65980.0,2,3,0.37,paid_ads,finance,employed,asia
1,71662.0,0,0,0.09,events,finance,employed,south_america
2,40220.0,2,5,0.55,referral,NA,employed,asia
3,88352.0,2,2,0.45,organic_search,finance,employed,north_america
4,89316.0,1,5,0.00,events,technology,employed,north_america
5,55610.0,2,3,0.38,paid_ads,healthcare,self_employed,europe
6,0.0,3,8,0.62,organic_search,retail,student,europe
7,63847.0,0,1,0.31,organic_search,technology,employed,south_america
8,102844.0,1,5,0.55,organic_search,technology,employed,north_america
9,48752.0,3,7,0.59,events,retail,employed,NA


In [34]:
df_train.shape

(3000, 8)

Question 3

In [ ]:
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[['lead_source', 'industry', 'employment_status', 'location']].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

The highest mutual information score is 0.03 for the "lead_source" variable